In [ ]:
pip install numpy pandas tensorflow scikit-learn matplotlib


### import the dataset

In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
sales_data = pd.read_csv('train.csv')
store_data = pd.read_csv('store.csv')

# Merge datasets
data = pd.merge(sales_data, store_data, on='Store', how='left')

# Preview the data
print(data.head())


### pre-process the data

In [ ]:
# Handle missing values
data.fillna(0, inplace=True)

# Convert Date column to datetime
data['Date'] = pd.to_datetime(data['Date'])

# Extract year, month, and day
data['Year'] = data['Date'].dt.year
data['Month'] = data['Date'].dt.month
data['Day'] = data['Date'].dt.day
data['DayOfWeek'] = data['Date'].dt.dayofweek

# Encode categorical variables
data['StateHoliday'] = data['StateHoliday'].astype('category').cat.codes
data['StoreType'] = data['StoreType'].astype('category').cat.codes
data['Assortment'] = data['Assortment'].astype('category').cat.codes

# Drop unnecessary columns
data.drop(['Date', 'Customers'], axis=1, inplace=True)


### split the data

In [ ]:
from sklearn.model_selection import train_test_split

# Feature matrix and target variable
X = data.drop('Sales', axis=1)
y = data['Sales']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
print(X_train.dtypes)


### scale the data

In [ ]:
!pip install pandas scikit-learn
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

### scale the data

In [ ]:
#Assuming 'data' is your DataFrame
# ... (your existing code for loading and preprocessing data) ...

# Feature matrix and target variable
X = data.drop('Sales', axis=1)
y = data['Sales']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create a OneHotEncoder for 'PromoInterval'
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore') # sparse=False for numpy array output

X_train['PromoInterval'] = X_train['PromoInterval'].astype(str)
X_test['PromoInterval'] = X_test['PromoInterval'].astype(str)

# Create a OneHotEncoder for 'PromoInterval'
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')  # sparse=False for numpy array output

# Fit and transform the 'PromoInterval' column in the training data
promo_interval_train = encoder.fit_transform(X_train[['PromoInterval']])

# Transform the 'PromoInterval' column in the testing data
promo_interval_test = encoder.transform(X_test[['PromoInterval']])

# Get feature names for the encoded columns
promo_interval_feature_names = encoder.get_feature_names_out(['PromoInterval'])

# Create DataFrames for the encoded 'PromoInterval'
promo_interval_train_df = pd.DataFrame(promo_interval_train, columns=promo_interval_feature_names, index=X_train.index)
promo_interval_test_df = pd.DataFrame(promo_interval_test, columns=promo_interval_feature_names, index=X_test.index)

# Drop the original 'PromoInterval' column and concatenate the encoded columns
X_train = X_train.drop('PromoInterval', axis=1).join(promo_interval_train_df)
X_test = X_test.drop('PromoInterval', axis=1).join(promo_interval_test_df)

# Now you can apply MinMaxScaler
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train = scaler_X.fit_transform(X_train)
X_test = scaler_X.transform(X_test)

y_train = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
y_test = scaler_y.transform(y_test.values.reshape(-1, 1))

### reshape the data

In [ ]:
X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

### build the model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Define the model
model = Sequential([
    LSTM(50, activation='relu', input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    Dense(1)
])

# Compile the model
model.compile(optimizer='adam', loss='mse')

# Summary of the model
model.summary()

### train the model

In [ ]:
# Train the model
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test), verbose=1)

### evalute the model

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Predictions
y_pred = model.predict(X_test)

# Inverse transform predictions and actuals
y_pred = scaler_y.inverse_transform(y_pred)
y_test = scaler_y.inverse_transform(y_test)

# Calculate metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")

### visuaize model

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(y_test[:100], label='Actual')
plt.plot(y_pred[:100], label='Predicted')
plt.title('Actual vs Predicted Sales')
plt.xlabel('Time')
plt.ylabel('Sales')
plt.legend()
plt.show()